# Gaussian Processes — Gaussian Processes for Machine Learning (2006)

_Papers · Classical ML_

**A distribution over functions: every prediction arrives with an honest error bar, computed in closed form from a kernel.**

---

This notebook is a practice scaffold for the **Gaussian Processes — Gaussian Processes for Machine Learning (2006)** lesson. We build GP regression one piece at a time: the kernel, the closed-form posterior via Cholesky, a check against scikit-learn, and a hand-recomputed 2x2 example. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy

### Step 1 — Make toy data and define the kernel

We sample 6 noisy points from `y = sin(x)`. The heart of a GP is its **kernel**: here the squared-exponential (RBF) `k(a,b) = sf2 · exp(-(a-b)² / 2ℓ²)`, which says two points have correlated function values when they are close. The length-scale `ℓ` sets how fast that correlation decays; `sf2` is the signal variance and `sn2` the observation noise.

In [ ]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel

rng = np.random.default_rng(0)

# ---- toy 1-D regression: y = sin(x) + noise at 6 inputs ----
Xtr = np.array([-4., -3., -1., 0., 2., 3.5])
ytr = np.sin(Xtr) + rng.normal(0, 0.1, Xtr.shape[0])
sf2, l, sn2 = 1.0, 1.0, 0.01          # signal var, length-scale, noise var

def rbf(A, B):                         # eq. (2.31), squared-exponential kernel
    d = A.reshape(-1, 1) - B.reshape(1, -1)
    return sf2 * np.exp(-(d ** 2) / (2 * l * l))

### Step 2 — Predict in closed form with Cholesky

This is Algorithm 2.1 from the book. We Cholesky-factor `K + sn2·I` (cheaper and more stable than inverting it), solve for `alpha = (K + sn2·I)⁻¹ y`, and then for any test points the **mean** is `k_*·alpha` (eq. 2.25) and the **variance** is the prior variance minus what the data explains (eq. 2.26). The square root of that variance is the error bar GPs are famous for.

In [ ]:
# ---- GP regression from scratch (Algorithm 2.1, via Cholesky) ----
def gp_predict(Xtest):
    K = rbf(Xtr, Xtr)
    L = np.linalg.cholesky(K + sn2 * np.eye(len(Xtr)))   # chol(K + sn2 I)
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, ytr))# (K+sn2 I)^-1 y
    Ks = rbf(Xtest, Xtr)                                 # k_* for each test pt
    mean = Ks @ alpha                                    # eq. (2.25)
    v = np.linalg.solve(L, Ks.T)
    var = sf2 - np.sum(v * v, axis=0)                    # eq. (2.26)
    return mean, np.sqrt(np.maximum(var, 1e-9))

Xte = np.linspace(-5, 5, 8)
mean_mine, std_mine = gp_predict(Xte)

### Step 3 — Check against scikit-learn

To trust our from-scratch code we compare it to scikit-learn's `GaussianProcessRegressor` using the *same* fixed kernel and noise (`alpha = sn2`, `optimizer=None` so it does not re-tune the hyperparameters). The mean and standard deviation should match to numerical precision — differences around `1e-16`.

In [ ]:
# ---- THE ORACLE: match sklearn's GaussianProcessRegressor ----
kern = ConstantKernel(sf2, "fixed") * RBF(l, "fixed")    # fixed -> no re-tuning
gp = GaussianProcessRegressor(kernel=kern, alpha=sn2, optimizer=None)  # alpha = sn2
gp.fit(Xtr.reshape(-1, 1), ytr)
mean_sk, std_sk = gp.predict(Xte.reshape(-1, 1), return_std=True)

print("mean allclose vs sklearn:", np.allclose(mean_mine, mean_sk, atol=1e-6))  # True
print("std  allclose vs sklearn:", np.allclose(std_mine,  std_sk,  atol=1e-6))  # True
print("max mean diff:", float(np.max(np.abs(mean_mine - mean_sk))))             # ~4e-16

### Step 4 — Recompute the 2x2 worked example

Finally we redo the lesson's hand example on just two points `x=[0,1]` with `y=[1,-1]`. The off-diagonal covariance `K01 = exp(-0.5) ≈ 0.6065`, the weights `alpha` come out equal-and-opposite, and at the test point `x*=0` we read off the mean and standard deviation. These printed numbers are the same ones the practice problems build on.

In [ ]:
# ---- recompute the 2x2 worked example ----
l2, sn2b = 1.0, 0.1
X2 = np.array([0., 1.])
y2 = np.array([1., -1.])

def k2(a, b):
    return np.exp(-(a - b) ** 2 / (2 * l2 * l2))

K2 = np.array([[k2(0, 0), k2(0, 1)], [k2(1, 0), k2(1, 1)]])
Ainv = np.linalg.inv(K2 + sn2b * np.eye(2))
a2 = Ainv @ y2
ks0 = np.array([k2(0, 0), k2(1, 0)])                     # test point x*=0

mean_at_0 = float(ks0 @ a2)
std_at_0 = float(np.sqrt(1 - ks0 @ Ainv @ ks0))
print("worked: K01 =", round(k2(0, 1), 4),               # 0.6065
      " alpha =", np.round(a2, 4),                       # [ 2.0265 -2.0265]
      " mean@0 =", round(mean_at_0, 4),                  # 0.7973
      " std@0 =", round(std_at_0, 4))                    # 0.2949

## Visualize the data & results

_Fit a GP (RBF kernel, length-scale ℓ=1) to 6 noisy points of sin(x). Where is the ±2σ uncertainty band narrow, and how does the band width between data points respond to the length-scale ℓ?_

### Step 1 — Reuse the data and a length-scale-parametrised fit

We recreate the same 6 noisy `sin` points, then wrap the whole Cholesky predictor in `fit_predict(l, Xte)` so we can refit with *any* length-scale `ℓ`. This makes the kernel's most important hyperparameter a knob we can turn to watch its effect on the uncertainty band.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

Xtr = np.array([-4., -3., -1., 0., 2., 3.5])
ytr = np.sin(Xtr) + rng.normal(0, 0.1, Xtr.shape[0])
sf2, sn2 = 1.0, 0.01

def fit_predict(l, Xte):
    def rbf(A, B):
        d = A.reshape(-1, 1) - B.reshape(1, -1)
        return sf2 * np.exp(-(d ** 2) / (2 * l * l))
    L = np.linalg.cholesky(rbf(Xtr, Xtr) + sn2 * np.eye(len(Xtr)))
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, ytr))
    Ks = rbf(Xte, Xtr)
    mean = Ks @ alpha
    v = np.linalg.solve(L, Ks.T)
    std = np.sqrt(np.maximum(sf2 - np.sum(v * v, axis=0), 1e-9))
    return mean, std

### Step 2 — Trace the posterior band on a dense grid

With `ℓ=1` we predict on a dense grid spanning the data and print the mean together with the `±2σ` band edges. The band should pinch tight right at the data points and fan out in the gaps and beyond the data, where the GP has less to go on.

In [ ]:
# top chart: posterior mean +/- 2 std on a dense grid (l=1)
Xte = np.linspace(-5.5, 5.5, 23)
m, s = fit_predict(1.0, Xte)
print("mean:", np.round(m, 3))
print("upper (m+2s):", np.round(m + 2 * s, 3))
print("lower (m-2s):", np.round(m - 2 * s, 3))

### Step 3 — Ablate the length-scale

Now we sweep `ℓ ∈ {0.3, 1.0, 3.0}` and read the predictive std at a gap point (`x=-2`) and far from the data (`x=5.3`). A short length-scale lets correlation decay fast, so uncertainty balloons in the gaps; a long one carries each point's influence far, keeping the band narrow. This reproduces the book's Figure 2.5.

In [ ]:
# bottom chart: length-scale ablation at a gap (x=-2) and far (x=5.3)
for l in [0.3, 1.0, 3.0]:
    _, ss = fit_predict(l, np.array([-2.0, 5.3]))
    print(f"l={l}: std@x=-2 = {ss[0]:.3f}, std@x=5.3 = {ss[1]:.3f}")

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Using the worked example (two points $x_1=0,x_2=1$, $\mathbf{y}=[1,-1]$, RBF $\sigma_f^2=1,\ell=1$, $\sigma_n^2=0.1$), predict the mean at the midpoint $x_*=0.5$. Predict the answer by symmetry before computing.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Test covariances: $k(0,0.5)=k(1,0.5)=\exp(-\tfrac12(0.5)^2)=\exp(-0.125)=0.8825$. — _$x_*$ is equidistant from both training points._
- Recall $\boldsymbol\alpha=[2.0265,-2.0265]$. — _Computed once in the worked example._
- Mean $=\mathbf{k}_*^\top\boldsymbol\alpha=0.8825\cdot 2.0265+0.8825\cdot(-2.0265)=0$. — _Equal-and-opposite contributions cancel._

**Answer:** The mean is exactly $0$. The two equally-distant training points have equal covariance with $x_*$, and their $\boldsymbol\alpha$ weights are equal and opposite (because $\mathbf{y}=[1,-1]$ is antisymmetric), so they cancel. The variance there is $\approx 0.087$ ($\sigma_*\approx 0.295$) — slightly larger than at the data points, reflecting that $x_*=0.5$ is a small gap between data.

</details>

**Problem 2.** In eq. (2.26), explain why the predictive variance equals the prior variance $k(\mathbf{x}_*,\mathbf{x}_*)$ when the test point is very far from all training points.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Far away, every $k(x_i,\mathbf{x}_*)\to 0$, so $\mathbf{k}_*\to\mathbf{0}$. — _RBF correlation decays to zero with distance._
- Then the subtracted term $\mathbf{k}_*^\top(K+\sigma_n^2 I)^{-1}\mathbf{k}_*\to 0$. — _A quadratic form in the zero vector is zero._
- So $\mathbb{V}[f_*]\to k(\mathbf{x}_*,\mathbf{x}_*)=\sigma_f^2$. — _Nothing is subtracted from the prior._

**Answer:** Far from data the test point is uncorrelated with every observation, so the data tells us nothing and the variance returns to the prior $\sigma_f^2$. This is exactly why the $\pm 2\sigma$ band fans out to its full prior width beyond the data — the GP is honest that it is just guessing there.

</details>

**Problem 3.** Ablation (length-scale). In the CODE/CODEVIZ, refit with length-scale $\ell=0.3$ and $\ell=3$ instead of $\ell=1$, keeping the data fixed. What happens to the uncertainty between data points, and why? (This reproduces the book's Figure 2.5.)

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Set $\ell=0.3$ and recompute the band on the dense grid. — _Short length-scale = correlation decays fast._
- With small $\ell$, $k(x_i,\mathbf{x}_*)$ drops to ~0 just a short distance from each data point. — _So $\mathbf{k}_*\approx\mathbf{0}$ even in small gaps between points._
- Set $\ell=3$ and recompute. — _Long length-scale = each point's influence reaches far._

**Answer:** With $\ell=0.3$ the band balloons to nearly the full prior width in every gap between data — the model assumes the function can wiggle fast, so it is uncertain the moment it leaves a data point. With $\ell=3$ each point's influence reaches across the whole input range, so the band stays narrow and the mean is a slowly-varying smooth curve. In our run the predictive std at the gap point $x=-2$ goes from $\approx 1.00$ ($\ell=0.3$) to $\approx 0.50$ ($\ell=1$) to $\approx 0.09$ ($\ell=3$). This is the length-scale's job: it controls how far confidence propagates from the data, matching Figure 2.5.

</details>